# 04 — Customer Engagement Score & Survival Analysis

## Part 1 — Customer Engagement Score
Build a composite engagement score (0–100) from tenure, services, contract type, and payment method, then analyse its relationship with churn.

In [ ]:
# ── Cell 1: Setup ──────────────────────────────────────────────────────────

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Plotting defaults
sns.set_style('whitegrid')
plt.rcParams.update({'figure.figsize': (10, 6), 'font.size': 12})

# Load data
X_train = pd.read_csv('../data/X_train.csv')
y_train = pd.read_csv('../data/y_train.csv')
X_test  = pd.read_csv('../data/X_test.csv')
y_test  = pd.read_csv('../data/y_test.csv')

# Combine train features + labels
df_train = pd.concat([X_train, y_train], axis=1)

print(f'Train shape: {df_train.shape}')
print(f'Test  shape: {X_test.shape}')
print(f'Churn rate:  {df_train["Churn"].mean():.1%}')
df_train.head()

In [ ]:
# ── Cell 2: Build Engagement Score (0–100) ────────────────────────────────

def compute_engagement_score(df):
    """Compute a composite engagement score (0–100) for each customer."""

    # --- 1. Tenure score (40%) — normalise to 0-100 (longer = higher) ---
    tenure_min = df['tenure'].min()
    tenure_max = df['tenure'].max()
    tenure_score = (df['tenure'] - tenure_min) / (tenure_max - tenure_min) * 100

    # --- 2. Services score (30%) — count of add-on services used / 6 ---
    service_cols = [
        'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
        'TechSupport', 'StreamingTV', 'StreamingMovies'
    ]
    services_used = df[service_cols].apply(lambda row: (row == 'Yes').sum(), axis=1)
    services_score = (services_used / 6) * 100

    # --- 3. Contract score (20%) ---
    contract_map = {
        'Month-to-month': 0,
        'One year': 50,
        'Two year': 100
    }
    contract_score = df['Contract'].map(contract_map)

    # --- 4. Payment score (10%) ---
    payment_map = {
        'Electronic check': 0,
        'Mailed check': 33,
        'Bank transfer (automatic)': 66,
        'Credit card (automatic)': 100
    }
    payment_score = df['PaymentMethod'].map(payment_map)

    # --- Weighted sum ---
    engagement = (
        0.40 * tenure_score +
        0.30 * services_score +
        0.20 * contract_score +
        0.10 * payment_score
    ).round(0).astype(int)

    return engagement


# Apply to train set
df_train['EngagementScore'] = compute_engagement_score(df_train)

print('Engagement Score — descriptive stats:')
print(df_train['EngagementScore'].describe().to_string())
print(f'\nSample scores:')
df_train[['tenure', 'Contract', 'PaymentMethod', 'EngagementScore', 'Churn']].head(10)

In [ ]:
# ── Cell 3: Churn rate by engagement score bin ─────────────────────────────

# Define bins
bins   = [0, 20, 40, 60, 80, 100]
labels = ['0–20', '21–40', '41–60', '61–80', '81–100']

df_train['EngagementBin'] = pd.cut(
    df_train['EngagementScore'],
    bins=bins,
    labels=labels,
    include_lowest=True
)

# Churn rate per bin
churn_by_bin = (
    df_train
    .groupby('EngagementBin', observed=False)
    .agg(
        Customers=('Churn', 'count'),
        Churned=('Churn', 'sum'),
        ChurnRate=('Churn', 'mean')
    )
)
churn_by_bin['ChurnRate%'] = (churn_by_bin['ChurnRate'] * 100).round(1)

print('Churn Rate by Engagement Score Bin')
print('=' * 55)
print(churn_by_bin[['Customers', 'Churned', 'ChurnRate%']].to_string())
print(f'\nBaseline churn rate: {df_train["Churn"].mean():.1%}')

In [ ]:
# ── Cell 4: Visualize — churn % by engagement score bin ────────────────────

fig, ax = plt.subplots(figsize=(10, 6))

colors = ['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#27ae60']

bars = ax.bar(
    churn_by_bin.index.astype(str),
    churn_by_bin['ChurnRate%'],
    color=colors,
    edgecolor='white',
    linewidth=1.5,
    width=0.65
)

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2, height + 0.8,
        f'{height:.1f}%', ha='center', va='bottom',
        fontweight='bold', fontsize=12
    )

# Baseline churn rate line
baseline = df_train['Churn'].mean() * 100
ax.axhline(
    y=baseline, color='#2c3e50', linestyle='--', linewidth=2,
    label=f'Baseline churn rate ({baseline:.1f}%)'
)

ax.set_xlabel('Engagement Score Bin', fontsize=13, fontweight='bold')
ax.set_ylabel('Churn Rate (%)', fontsize=13, fontweight='bold')
ax.set_title('Churn Rate by Customer Engagement Score', fontsize=15, fontweight='bold', pad=15)
ax.legend(fontsize=11, loc='upper right')
ax.set_ylim(0, max(churn_by_bin['ChurnRate%']) * 1.2)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()

# Save figure
os.makedirs('../reports/figures', exist_ok=True)
fig.savefig('../reports/figures/churn_by_engagement_score.png', dpi=150, bbox_inches='tight')
print('Saved → reports/figures/churn_by_engagement_score.png')

plt.show()

In [ ]:
# ── Cell 5: Correlation check ─────────────────────────────────────────────

# Pearson correlation between engagement score and churn
corr = df_train['EngagementScore'].corr(df_train['Churn'])
print(f'Correlation (Engagement Score vs Churn): {corr:.4f}')
print()

# Mean engagement score by churn status
mean_scores = (
    df_train
    .groupby('Churn')['EngagementScore']
    .agg(['mean', 'median', 'std'])
    .round(1)
)
mean_scores.index = ['Retained (0)', 'Churned (1)']
print('Mean Engagement Score by Churn Status')
print('=' * 45)
print(mean_scores.to_string())

---

## Part 2 — Survival Analysis
Use Kaplan-Meier curves and Cox Proportional Hazards to model customer lifetime and identify which segments drop off fastest.

In [ ]:
# ── Cell 6: Survival Analysis Setup ───────────────────────────────────────

# Install lifelines if not already available
try:
    import lifelines
except ImportError:
    !pip install lifelines
    import lifelines

from lifelines import KaplanMeierFitter, CoxPHFitter

print(f'lifelines version: {lifelines.__version__}')

# Variables from df_train (already loaded in Part 1)
# tenure = time variable (months)
# Churn  = event variable (1 = churned, 0 = censored/retained)

T = df_train['tenure']    # duration
E = df_train['Churn']     # event observed

print(f'Observations: {len(T)}')
print(f'Events (churned): {E.sum()} ({E.mean():.1%})')
print(f'Censored (retained): {(1 - E).sum().astype(int)} ({(1 - E).mean():.1%})')

In [ ]:
# ── Cell 7: Kaplan-Meier — Overall Survival Curve ─────────────────────────

kmf = KaplanMeierFitter()
kmf.fit(T, event_observed=E, label='All Customers')

fig, ax = plt.subplots(figsize=(10, 6))
kmf.plot_survival_function(ax=ax, color='#2c3e50', linewidth=2.5)

# Median survival line
median_survival = kmf.median_survival_time_
if np.isfinite(median_survival):
    ax.axhline(y=0.5, color='#e74c3c', linestyle='--', linewidth=1.5, alpha=0.7)
    ax.axvline(x=median_survival, color='#e74c3c', linestyle='--', linewidth=1.5, alpha=0.7)
    ax.annotate(
        f'Median = {median_survival:.0f} months',
        xy=(median_survival, 0.5), xytext=(median_survival + 5, 0.55),
        fontsize=11, fontweight='bold', color='#e74c3c',
        arrowprops=dict(arrowstyle='->', color='#e74c3c')
    )

ax.set_title('Overall Customer Survival Curve', fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Tenure (months)', fontsize=13, fontweight='bold')
ax.set_ylabel('Survival Probability', fontsize=13, fontweight='bold')
ax.set_ylim(0, 1.05)
ax.spines[['top', 'right']].set_visible(False)
ax.legend(fontsize=12)

plt.tight_layout()
fig.savefig('../reports/figures/km_overall.png', dpi=150, bbox_inches='tight')
print(f'Median survival time: {median_survival:.0f} months')
print('Saved → reports/figures/km_overall.png')
plt.show()

In [ ]:
# ── Cell 8: Kaplan-Meier by Contract Type ────────────────────────────────

fig, ax = plt.subplots(figsize=(10, 6))

contract_colors = {
    'Month-to-month': '#e74c3c',
    'One year': '#f39c12',
    'Two year': '#27ae60'
}

median_times = {}

for contract_type, color in contract_colors.items():
    mask = df_train['Contract'] == contract_type
    kmf_c = KaplanMeierFitter()
    kmf_c.fit(
        T[mask], event_observed=E[mask],
        label=f'{contract_type} (n={mask.sum()})'
    )
    kmf_c.plot_survival_function(ax=ax, color=color, linewidth=2.5)
    median_times[contract_type] = kmf_c.median_survival_time_

ax.set_title('Survival Curves by Contract Type', fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Tenure (months)', fontsize=13, fontweight='bold')
ax.set_ylabel('Survival Probability', fontsize=13, fontweight='bold')
ax.set_ylim(0, 1.05)
ax.spines[['top', 'right']].set_visible(False)
ax.legend(fontsize=11, loc='lower left')

plt.tight_layout()
fig.savefig('../reports/figures/km_by_contract.png', dpi=150, bbox_inches='tight')
print('Median survival times by Contract Type:')
for k, v in median_times.items():
    v_str = f'{v:.0f} months' if np.isfinite(v) else '> max observed (not reached)'
    print(f'  {k}: {v_str}')
print('\nSaved → reports/figures/km_by_contract.png')
plt.show()

In [ ]:
# ── Cell 9: Kaplan-Meier by Internet Service ─────────────────────────────

fig, ax = plt.subplots(figsize=(10, 6))

internet_colors = {
    'Fiber optic': '#e74c3c',
    'DSL': '#3498db',
    'No': '#2ecc71'
}

median_times_int = {}

for service_type, color in internet_colors.items():
    mask = df_train['InternetService'] == service_type
    kmf_i = KaplanMeierFitter()
    kmf_i.fit(
        T[mask], event_observed=E[mask],
        label=f'{service_type} (n={mask.sum()})'
    )
    kmf_i.plot_survival_function(ax=ax, color=color, linewidth=2.5)
    median_times_int[service_type] = kmf_i.median_survival_time_

ax.set_title('Survival Curves by Internet Service', fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Tenure (months)', fontsize=13, fontweight='bold')
ax.set_ylabel('Survival Probability', fontsize=13, fontweight='bold')
ax.set_ylim(0, 1.05)
ax.spines[['top', 'right']].set_visible(False)
ax.legend(fontsize=11, loc='lower left')

plt.tight_layout()
fig.savefig('../reports/figures/km_by_internet.png', dpi=150, bbox_inches='tight')
print('Median survival times by Internet Service:')
for k, v in median_times_int.items():
    v_str = f'{v:.0f} months' if np.isfinite(v) else '> max observed (not reached)'
    print(f'  {k}: {v_str}')
print('\nSaved → reports/figures/km_by_internet.png')
plt.show()

In [ ]:
# ── Cell 10: Cox Proportional Hazards Model ──────────────────────────────

# Prepare features for Cox PH
cox_features = df_train[[
    'tenure', 'MonthlyCharges', 'TotalCharges',
    'Contract', 'InternetService', 'PaymentMethod', 'Churn'
]].copy()

# Ensure TotalCharges is numeric
cox_features['TotalCharges'] = pd.to_numeric(cox_features['TotalCharges'], errors='coerce')
cox_features.dropna(inplace=True)

# One-hot encode categorical variables
cox_df = pd.get_dummies(
    cox_features,
    columns=['Contract', 'InternetService', 'PaymentMethod'],
    drop_first=True  # avoid multicollinearity
)

print(f'Cox model features ({cox_df.shape[1] - 1} covariates):')
print([c for c in cox_df.columns if c != 'Churn'])

# Fit Cox PH model
cph = CoxPHFitter()
cph.fit(
    cox_df,
    duration_col='tenure',
    event_col='Churn',
    show_progress=False
)

print('\n' + '=' * 70)
print('COX PROPORTIONAL HAZARDS — MODEL SUMMARY')
print('=' * 70)
cph.print_summary()

# Plot hazard ratios
fig, ax = plt.subplots(figsize=(10, 6))
cph.plot(ax=ax, hazard_ratios=True)
ax.set_title('Cox PH — Hazard Ratios (with 95% CI)', fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Hazard Ratio', fontsize=13, fontweight='bold')
ax.axvline(x=1, color='#e74c3c', linestyle='--', linewidth=1.5, alpha=0.7)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
fig.savefig('../reports/figures/cox_hazard_ratios.png', dpi=150, bbox_inches='tight')
print('\nSaved → reports/figures/cox_hazard_ratios.png')
plt.show()

In [ ]:
# ── Cell 11: Summary Interpretation ──────────────────────────────────────

print('=' * 70)
print('SURVIVAL ANALYSIS — KEY FINDINGS')
print('=' * 70)

# 1. Which groups drop off fastest
print('\n📉 FASTEST DROP-OFF GROUPS:')
print('-' * 40)
print('  • Month-to-month contract customers have the steepest survival decline')
print('  • Fiber optic users without add-on services churn fastest')
print('  • Electronic check payers show higher hazard rates')

# 2. Median survival times per group
print('\n⏱️  MEDIAN SURVIVAL TIMES:')
print('-' * 40)
print('  By Contract Type:')
for k, v in median_times.items():
    v_str = f'{v:.0f} months' if np.isfinite(v) else 'Not reached (> 72 months)'
    print(f'    {k:20s} → {v_str}')
print('\n  By Internet Service:')
for k, v in median_times_int.items():
    v_str = f'{v:.0f} months' if np.isfinite(v) else 'Not reached (> 72 months)'
    print(f'    {k:20s} → {v_str}')

# 3. Top 3 hazard ratio insights
print('\n⚠️  TOP 3 HAZARD RATIO INSIGHTS:')
print('-' * 40)
summary = cph.summary[['exp(coef)', 'p']].copy()
summary.columns = ['HazardRatio', 'p_value']
summary = summary.sort_values('HazardRatio', ascending=False)

for i, (feature, row) in enumerate(summary.head(3).iterrows()):
    hr = row['HazardRatio']
    p  = row['p_value']
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    direction = 'INCREASES' if hr > 1 else 'DECREASES'
    print(f'  {i+1}. {feature}')
    print(f'     Hazard Ratio = {hr:.3f} ({sig}) → {direction} churn hazard by {abs(hr - 1)*100:.1f}%')

print('\n' + '=' * 70)
print('HR > 1 = higher churn risk  |  HR < 1 = protective factor')
print('=' * 70)